In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import FastSAM
import matplotlib.pyplot as plt
import cv2
model = FastSAM("FastSAM-s.pt")  # or FastSAM-x.pt



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

source="/content/owl.png"

def visualize_fastsam_results(image_path, results):
    # Read image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Get image dimensions
    h, w = image.shape[:2]

    # Create figure
    plt.figure(figsize=(10, 10))

    # Plot original image
    plt.imshow(image)

    # Get masks from results
    if hasattr(results[0], 'masks') and results[0].masks is not None:
        masks = results[0].masks.data.cpu().numpy()
        # Overlay each mask
        for mask in masks:
            # Resize mask to match image dimensions
            resized_mask = cv2.resize(mask.astype(float), (w, h), interpolation=cv2.INTER_NEAREST)

            # Create colored mask overlay
            colored_mask = np.zeros_like(image)
            colored_mask[resized_mask > 0] = [255, 0, 0]  # Red color for mask
            plt.imshow(colored_mask, alpha=0.5)  # Adjust alpha for transparency

    plt.axis('off')
    plt.show()

# Use this to visualize
results = model(source, bboxes=[439, 437, 524, 709])
visualize_fastsam_results(source, results)


results

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
source = "/content/cat.jpg"

def visualize_point_prompt_results(image_path, results):
    # Read image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Create figure
    plt.figure(figsize=(10, 10))

    # Plot original image
    plt.imshow(image)

    # Get masks from results
    if hasattr(results[0], 'masks') and results[0].masks is not None:
        masks = results[0].masks.data.cpu().numpy()
        # Create colored overlay for masks
        mask_overlay = np.zeros_like(image)
        for mask in masks:
            # Resize mask to match image dimensions
            resized_mask = cv2.resize(mask, (image.shape[1], image.shape[0]))
            mask_overlay[resized_mask > 0] = [0, 255, 0]  # Green color for mask
        plt.imshow(mask_overlay, alpha=0.5)

    # Plot the point
    plt.scatter(555, 300, c='red', s=100)  # Red dot for the point prompt

    plt.axis('off')
    plt.show()



In [ ]:

# Run inference with point prompt
results = model(source, points=[[555, 300]], labels=[1])
visualize_point_prompt_results(source, results)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def extract_segmented_object(image_path, results, invert=False):
    """
    Extract a segmented object from an image using a mask.

    Parameters:
        image_path (str): Path to the input image.
        results (object): Inference results containing the mask.
        invert (bool): If True, invert the selection (keep everything except the mask).

    Returns:
        np.ndarray: The resulting image with transparency.
    """
    # Read image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    if hasattr(results[0], 'masks') and results[0].masks is not None:
        # Get the first mask (assuming it's the most relevant one)
        mask = results[0].masks.data[0].cpu().numpy()

        # Resize mask to match image dimensions
        resized_mask = cv2.resize(mask, (image.shape[1], image.shape[0]))

        # Create a boolean mask
        bool_mask = resized_mask > 0

        # Invert the mask if required
        if invert:
            bool_mask = ~bool_mask

        # Create output image with transparent background
        output = np.zeros((image.shape[0], image.shape[1], 4), dtype=np.uint8)

        # Copy RGB channels where mask is True
        output[bool_mask, :3] = image[bool_mask]

        # Set alpha channel where mask is True
        output[bool_mask, 3] = 255

        # Display result
        plt.figure(figsize=(10, 10))
        plt.imshow(output)
        plt.axis('off')
        plt.show()

        return output
    else:
        print("No mask found in results")
        return None

# Example Usage
results = model(source, points=[[500, 200]], labels=[1])

# Normal extraction
cutout = extract_segmented_object(source, results)

# Inverted selection
cutout_inverted = extract_segmented_object(source, results, invert=True)

# Save with transparency
if cutout is not None:
    plt.imsave('owl_cutout.png', cutout, format='png')
if cutout_inverted is not None:
    plt.imsave('owl_cutout_inverted.png', cutout_inverted, format='png')


In [ ]:
from ultralytics import FastSAM
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Define source image
source = "/content/cat.jpg"

# Create FastSAM model
model = FastSAM("FastSAM-s.pt")

# Run inference with text prompt
results = model(source, texts=["Sofa"])  # Ensure the text prompt is a list

# Visualization function
def visualize_text_prompt_results(image_path, results):
    """
    Visualize the segmentation results with text prompt.

    Parameters:
        image_path (str): Path to the input image.
        results (object): Inference results from FastSAM.
    """
    # Read image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Create figure
    plt.figure(figsize=(10, 10))

    # Plot original image
    plt.imshow(image)

    # Get masks from results
    if hasattr(results[0], 'masks') and results[0].masks is not None:
        masks = results[0].masks.data.cpu().numpy()

        # Resize masks to match image dimensions
        mask_overlay = np.zeros_like(image)
        for mask in masks:
            resized_mask = cv2.resize(mask, (image.shape[1], image.shape[0]))
            mask_overlay[resized_mask > 0] = [255, 0, 0]  # Red color for mask

        # Overlay the masks on the image
        plt.imshow(mask_overlay, alpha=0.5)

    plt.axis('off')
    plt.show()

# Visualize results
visualize_text_prompt_results(source, results)
